# The Agentic Paradigm: Workflow Architecture & Use Cases
As an Architect, your primary job is to pick the right tool for the job. Beginners try to solve every problem with a longer prompt. Seniors design **Workflows**. 

Today, we define the two pillars of Agentic Logic:
1. **RAG (Knowledge-Driven):** Use this when the AI needs to "Read" private data to answer a question.
2. **Tools (Action-Driven):** Use this when the AI needs to "Act" on the world (API calls, Database writes, Calculations).

An **Agent** is simply a loop where the LLM decides which pillar to use at any given moment.

In [1]:
import os
import json
from openai import OpenAI
from litellm import completion
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)
client = OpenAI()

print("🤖 Agentic Architecture Environment Ready!")

🤖 Agentic Architecture Environment Ready!


## 1. The Architect's Decision Matrix

| Scenario | Primary Pattern | Why? |
| :--- | :--- | :--- |
| "What is our refund policy?" | **RAG** | Static knowledge retrieval. |
| "Update the customer's address." | **Tools** | Action required in a database. |
| "Check stock and email the user." | **Agent (Loop)** | Needs to read (Action 1) then act (Action 2). |
| "Summarize the 2024 tax code." | **RAG** | Large document processing. |

## 2. Implementing the "CRM Architect" Agent

We will build a simple agent that can either **Search** a knowledge base (RAG) or **Update** a customer record (Tool). 

In [2]:
# Mock Data Stores
MOCK_KNOWLEDGE_BASE = {"refund_policy": "Refunds are allowed within 30 days with a receipt."}
MOCK_CRM = {"user_123": {"name": "Alice", "status": "Active"}}

# 1. Define the Tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_knowledge_base",
            "description": "Search internal documentation for policies.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "update_crm_status",
            "description": "Update a user's status in the CRM database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_id": {"type": "string"},
                    "new_status": {"type": "string"}
                }
            }
        }
    }
]

# 2. Define the Execution Functions
def search_knowledge_base(query):
    print(f"🔍 RAG: Searching for {query}...")
    return MOCK_KNOWLEDGE_BASE.get(query, "Information not found.")

def update_crm_status(user_id, new_status):
    print(f"💾 TOOL: Updating {user_id} to {new_status}...")
    MOCK_CRM[user_id]["status"] = new_status
    return "Success: CRM Updated."

print("🏗️ Agent Components Defined.")

🏗️ Agent Components Defined.


## 3. The Orchestration Loop

The Architect builds the code that handles the "Wait-Execute-Report" cycle. 

In [4]:
def run_agent(user_query):
    messages = [
        {"role": "system", "content": "You are a Customer Operations Agent. Use tools to help the user."},
        {"role": "user", "content": user_query}
    ]
    
    # --- TURN 1: The Model Decides ---
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools
    )
    
    msg = response.choices[0].message
    
    # --- TURN 2: Code Execution (The Architect's Responsibility) ---
    if msg.tool_calls:
        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            
            # Execute the correct logic
            if name == "search_knowledge_base":
                result = search_knowledge_base(args['query'])
            elif name == "update_crm_status":
                result = update_crm_status(args['user_id'], args['new_status'])
            
            # Append the result back to the conversation
            messages.append(msg)
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
            
        # --- TURN 3: Final Answer ---
        final_response = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
        return final_response.choices[0].message.content
    
    return msg.content

print("--- Agent Testing ---")
print(f"Query 1: {run_agent('What is refund policy?')}")
print(f"\nQuery 2: {run_agent('Set user_123 to VIP status.')}")
print(f"\nFinal CRM State: {MOCK_CRM}")

--- Agent Testing ---
🔍 RAG: Searching for refund policy...
Query 1: Unfortunately, I couldn't find specific details about the refund policy in the knowledge base. However, I can provide you with a general idea. 

Typically, a refund policy outlines the conditions under which customers can return products and receive their money back. Key points often include:

1. **Time Frame**: Most policies specify a period (e.g., 30 days) during which returns are accepted.
2. **Condition of the Product**: Items usually need to be in their original condition, unopened, or unused.
3. **Proof of Purchase**: A receipt or order confirmation may be required.
4. **Exclusions**: Some items like personalized products or perishables may not be eligible for refunds.
5. **Process**: Instructions on how to initiate a return and receive a refund.

I recommend checking the specific refund policy for the company you're inquiring about on their official website or contacting their customer service for detailed info